# RNN vs LSTM vs GRU Text Classification Comparison (PyTorch)

In this notebook, we will use the **IMDB Movie Review dataset** to compare three fundamental Recurrent Neural Network architectures in PyTorch:
1. **Vanilla RNN (Recurrent Neural Network)**
2. **LSTM (Long Short-Term Memory)**
3. **GRU (Gated Recurrent Unit)**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 1. Load the IMDB dataset (using Keras helper to fetch the raw numpy arrays easily)
vocab_size = 10000
max_length = 100

(X_train_seq, y_train_seq), (X_test_seq, y_test_seq) = imdb.load_data(num_words=vocab_size)

# Pad sequences to ensure uniform input length
X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post')

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train_pad, dtype=torch.long)
y_train_t = torch.tensor(y_train_seq, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test_pad, dtype=torch.long)
y_test_t = torch.tensor(y_test_seq, dtype=torch.float32).unsqueeze(1)

# Create DataLoaders
batch_size = 128
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

# Split train dataset into train and validation (80-20 split)
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_sub, val_sub = torch.utils.data.random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_sub, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_sub, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f'Train size: {len(train_sub)}, Validation size: {len(val_sub)}, Test size: {len(test_dataset)}')

Using device: cpu
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train size: 20000, Validation size: 5000, Test size: 25000


## Define the PyTorch Model Architecture

We will define a single configurable PyTorch model class that can instantiate a vanilla **RNN**, an **LSTM**, or a **GRU** based on the architecture string passed to it.

In [2]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, rnn_type='LSTM'):
        super(RNNClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn_type = rnn_type

        if rnn_type == 'RNN':
            self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        elif rnn_type == 'LSTM':
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        elif rnn_type == 'GRU':
            self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        embedded = self.embedding(x) # shape: (batch_size, seq_len, embedding_dim)

        if self.rnn_type == 'LSTM':
            # LSTM returns output, (hn, cn)
            out, (hn, cn) = self.rnn(embedded)
        else:
            # RNN and GRU return output, hn
            out, hn = self.rnn(embedded)

        # Use the hidden state of the final time step
        # hn shape: (num_layers, batch_size, hidden_dim)
        last_hidden = hn[-1]

        out = self.fc(last_hidden)
        return self.sigmoid(out)

## Training and Evaluation Helper Functions

In [3]:
def train_model(model, train_loader, val_loader, epochs=5, lr=0.001):
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            preds = (outputs >= 0.5).float()
            correct_train += (preds == labels).sum().item()
            total_train += labels.size(0)

        epoch_train_loss = running_loss / total_train
        epoch_train_acc = correct_train / total_train

        # Validation phase
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                preds = (outputs >= 0.5).float()
                correct_val += (preds == labels).sum().item()
                total_val += labels.size(0)

        epoch_val_loss = val_loss / total_val
        epoch_val_acc = correct_val / total_val

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)

        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_train_loss:.4f} - Acc: {epoch_train_acc:.4f} - Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc:.4f}")

    return history

def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            preds = (outputs >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

## Train and Compare all Three Architectures (RNN, LSTM, GRU)

In [4]:
embedding_dim = 32
hidden_dim = 32
epochs = 5

# 1. Train Vanilla RNN
print("=== Training RNN ===")
rnn_model = RNNClassifier(vocab_size, embedding_dim, hidden_dim, rnn_type='RNN').to(device)
rnn_history = train_model(rnn_model, train_loader, val_loader, epochs=epochs)

# 2. Train LSTM
print("\n=== Training LSTM ===")
lstm_model = RNNClassifier(vocab_size, embedding_dim, hidden_dim, rnn_type='LSTM').to(device)
lstm_history = train_model(lstm_model, train_loader, val_loader, epochs=epochs)

# 3. Train GRU
print("\n=== Training GRU ===")
gru_model = RNNClassifier(vocab_size, embedding_dim, hidden_dim, rnn_type='GRU').to(device)
gru_history = train_model(gru_model, train_loader, val_loader, epochs=epochs)

=== Training RNN ===
Epoch 1/5 - Loss: 0.6863 - Acc: 0.5434 - Val Loss: 0.6764 - Val Acc: 0.5664
Epoch 2/5 - Loss: 0.6563 - Acc: 0.6084 - Val Loss: 0.6463 - Val Acc: 0.6184
Epoch 3/5 - Loss: 0.6458 - Acc: 0.6377 - Val Loss: 0.6826 - Val Acc: 0.5592
Epoch 4/5 - Loss: 0.6632 - Acc: 0.5970 - Val Loss: 0.6386 - Val Acc: 0.6202
Epoch 5/5 - Loss: 0.6067 - Acc: 0.6643 - Val Loss: 0.6162 - Val Acc: 0.6650

=== Training LSTM ===
Epoch 1/5 - Loss: 0.6871 - Acc: 0.5403 - Val Loss: 0.6742 - Val Acc: 0.5772
Epoch 2/5 - Loss: 0.6231 - Acc: 0.6546 - Val Loss: 0.5732 - Val Acc: 0.7128
Epoch 3/5 - Loss: 0.5289 - Acc: 0.7444 - Val Loss: 0.5159 - Val Acc: 0.7536
Epoch 4/5 - Loss: 0.4808 - Acc: 0.7782 - Val Loss: 0.4874 - Val Acc: 0.7746
Epoch 5/5 - Loss: 0.4167 - Acc: 0.8196 - Val Loss: 0.4718 - Val Acc: 0.7828

=== Training GRU ===
Epoch 1/5 - Loss: 0.6840 - Acc: 0.5492 - Val Loss: 0.6665 - Val Acc: 0.5912
Epoch 2/5 - Loss: 0.6029 - Acc: 0.6732 - Val Loss: 0.5577 - Val Acc: 0.7180
Epoch 3/5 - Loss: 0.48